# Domain C: Contextual Adaptation and History Dependence

This notebook addresses three questions:
- **C1:** Do responses differ between the first and second presentation of the same block type (context-dependent adaptation)?
- **C2:** Does the context (contrast-varying vs. speed-varying) alter tuning curves?
- **C3:** Do responses change across days (multi-day representational drift)?

**Data:** Zarr multimodal stores with ΔF/F calcium traces (X matrix, cells × trials), stimulus metadata (var), cell-type labels & gene expression (obs).
Context structure: Blocks 0,2 = contrast-context (TF fixed at 1 Hz); Blocks 1,3 = speed-context (contrast fixed at 0.8).

In [ ]:
import sys
sys.path.insert(0, '.')

import numpy as np
import pandas as pd
import zarr
import matplotlib.pyplot as plt
import seaborn as sns
from types import SimpleNamespace
from scipy.stats import kruskal, wilcoxon, mannwhitneyu, spearmanr, pearsonr, ttest_rel
from scipy.optimize import curve_fit
import warnings
warnings.filterwarnings('ignore')

from functions.data_loading import load_mouse_zarr, load_zarr_10hz
from functions.analysis import get_block_responses, compute_adaptation_index, get_trial_position_responses, exp_decay, linear_CKA
from functions.tuning import naka_rushton, normalization_model

sns.set_context('talk')
sns.set_style('whitegrid')

ZARR_DIR = 'multimodal_data'
MOUSE_IDS = ['778174', '786297', '797371']
SESSIONS = ['session_1', 'session_2', 'session_3']

adata_list = [load_mouse_zarr(mid, zarr_dir=ZARR_DIR) for mid in MOUSE_IDS]

obs = pd.concat([a.obs for a in adata_list], ignore_index=True)
X = np.vstack([a.X for a in adata_list])
var = adata_list[0].var.copy()

print(f"Total cells: {len(obs)}, Total trials: {X.shape[1]}")

# Backward-compatible adata object for downstream cells
adata = SimpleNamespace(X=X, obs=obs, var=var, n_obs=len(obs), n_vars=X.shape[1])

# ── Subclass setup ──
SUBCLASS_ORDER = ['007 L2/3 IT CTX Glut', '006 L4/5 IT CTX Glut', '022 L5 ET CTX Glut',
                  '052 Pvalb Gaba', '053 Sst Gaba', '046 Vip Gaba', '049 Lamp5 Gaba']
SUBCLASS_COLORS = {'007 L2/3 IT CTX Glut': '#1f77b4', '006 L4/5 IT CTX Glut': '#2ca02c',
                   '022 L5 ET CTX Glut': '#9467bd', '052 Pvalb Gaba': '#d62728',
                   '053 Sst Gaba': '#ff7f0e', '046 Vip Gaba': '#e377c2', '049 Lamp5 Gaba': '#8c564b'}
SUBCLASS_SHORT = {'007 L2/3 IT CTX Glut': 'L2/3 IT', '006 L4/5 IT CTX Glut': 'L4/5 IT',
                  '022 L5 ET CTX Glut': 'L5 ET', '052 Pvalb Gaba': 'Pvalb',
                  '053 Sst Gaba': 'Sst', '046 Vip Gaba': 'Vip', '049 Lamp5 Gaba': 'Lamp5'}
present_subclasses = [s for s in SUBCLASS_ORDER if s in obs['subclass_name'].unique()]
short_order = [SUBCLASS_SHORT[s] for s in present_subclasses]
short_pal = {SUBCLASS_SHORT[k]: v for k, v in SUBCLASS_COLORS.items()}

orientations = np.array([0, 45, 90, 135, 180, 225, 270, 315])
contrasts = np.array([0.05, 0.1, 0.2, 0.4, 0.8])
tfs = np.array([1, 2, 4, 8, 15])

## C4: Within-Trial Temporal Adaptation (10 Hz ΔF/F)

Using 10 Hz trial-resolved traces, we examine how the temporal shape of responses changes:
- **C4.1** How does the PSTH shape evolve from the first to the last trial within a block? (within-block habituation)
- **C4.2** Does the transient/sustained ratio shift between contrast-context vs speed-context blocks?

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# C4.1  Load 10 Hz data & examine within-block PSTH evolution
# ══════════════════════════════════════════════════════════════════════

# Use one representative mouse (largest)
mouse_pick = obs['mouse_id'].value_counts().idxmax()
pk = load_zarr_10hz(mouse_pick, zarr_dir=ZARR_DIR)
dff_10hz = pk['dff']            # (n_trials, 41, n_cells)
ti_10hz = pk['trial_info']
time_rel = pk['time_rel']       # (41,) from -1s to 3s
uids_10hz = pk['unique_ids']
n_trials_pk, n_tp, n_cells_pk = dff_10hz.shape

# Match cells to obs for subclass labels
obs_mouse = obs[obs['mouse_id'] == mouse_pick]
cell_sc_map = {}
for ci, uid in enumerate(uids_10hz):
    match = obs_mouse[obs_mouse['unique_id'] == uid]
    if len(match) > 0:
        cell_sc_map[ci] = match.iloc[0]['subclass_name']

matched_cells = sorted(cell_sc_map.keys())
cell_subclasses = np.array([cell_sc_map[ci] for ci in matched_cells])

# Identify block boundaries in trial_info
non_gray = ~ti_10hz['gray_screen'].astype(bool)
block_ids = ti_10hz.loc[non_gray.values, 'stim_block'].values
grating_trial_idx = np.where(non_gray.values)[0]

print(f"Mouse {mouse_pick}: {n_cells_pk} cells, {n_trials_pk} total trials, {len(grating_trial_idx)} grating trials")
print(f"Matched {len(matched_cells)} cells to taxonomy")

# ── Within-block temporal evolution ──
# Split each block's grating trials into early (first 20%) vs late (last 20%)
stim_idx = (time_rel >= 0) & (time_rel <= 2.0)
early_idx_t = (time_rel >= 0) & (time_rel <= 0.5)
late_idx_t = (time_rel >= 1.5) & (time_rel <= 2.0)

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
block_names = {0: 'Block 0 (Contrast-ctx)', 1: 'Block 1 (Speed-ctx)',
               2: 'Block 2 (Contrast-ctx)', 3: 'Block 3 (Speed-ctx)'}

for bi, block in enumerate([0, 1, 2, 3]):
    ax = axes.flat[bi]
    block_trial_mask = block_ids == block
    block_trials = grating_trial_idx[block_trial_mask]
    n_bt = len(block_trials)
    n_fifth = max(1, n_bt // 5)
    
    early_trials = block_trials[:n_fifth]
    late_trials = block_trials[-n_fifth:]
    
    for sc in present_subclasses:
        sc_cells = [matched_cells[i] for i in range(len(matched_cells)) if cell_subclasses[i] == sc]
        if len(sc_cells) < 3:
            continue
        dff_e = np.nanmean(dff_10hz[early_trials][:, :, sc_cells], axis=(0, 2))
        dff_l = np.nanmean(dff_10hz[late_trials][:, :, sc_cells], axis=(0, 2))
        ax.plot(time_rel, dff_e, color=SUBCLASS_COLORS[sc], linewidth=2, label=f'{SUBCLASS_SHORT[sc]} early')
        ax.plot(time_rel, dff_l, color=SUBCLASS_COLORS[sc], linewidth=2, ls='--', label=f'{SUBCLASS_SHORT[sc]} late')
    
    ax.axvline(0, color='k', ls='--', alpha=0.3)
    ax.set_title(block_names[block], fontweight='bold')
    ax.set_xlabel('Time (s)')
    ax.set_ylabel('ΔF/F')
    if bi == 0:
        ax.legend(fontsize=6, ncol=2)

plt.suptitle('C4: Within-Block PSTH Adaptation (Early vs Late Trials)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# ── Per-subclass: PSTH shape comparison (contrast vs speed context) ──
present_sc_b = [s for s in present_subclasses if sum(1 for c in matched_cells if cell_sc_map.get(c) == s) >= 3]
fig, axes = plt.subplots(len(present_sc_b), 2, figsize=(14, 4*len(present_sc_b)))
if len(present_sc_b) == 1:
    axes = axes.reshape(1, -1)

for ri, sc in enumerate(present_sc_b):
    sc_cells = [matched_cells[i] for i in range(len(matched_cells)) if cell_subclasses[i] == sc]
    if len(sc_cells) < 3:
        continue
    
    for ci_block, (block_label, block_val) in enumerate([('Contrast-ctx (0,2)', [0, 2]),
                                                          ('Speed-ctx (1,3)', [1, 3])]):
        ax = axes[ri, ci_block]
        bm = np.isin(block_ids, block_val)
        bt = grating_trial_idx[bm]
        n_bt = len(bt)
        n_fifth = max(1, n_bt // 5)
        
        dff_e = np.nanmean(dff_10hz[bt[:n_fifth]][:, :, sc_cells], axis=(0, 2))
        dff_l = np.nanmean(dff_10hz[bt[-n_fifth:]][:, :, sc_cells], axis=(0, 2))
        
        ax.plot(time_rel, dff_e, color=SUBCLASS_COLORS[sc], linewidth=2, label='Early')
        ax.plot(time_rel, dff_l, color=SUBCLASS_COLORS[sc], linewidth=2, ls='--', label='Late')
        ax.axvline(0, color='k', ls='--', alpha=0.3)
        ax.set_title(f'{SUBCLASS_SHORT[sc]} — {block_label}', fontweight='bold',
                     color=SUBCLASS_COLORS[sc], fontsize=10)
        ax.set_ylabel('ΔF/F')
        ax.legend(fontsize=7)

plt.suptitle('C4: Within-Block PSTH Adaptation by Subclass', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# C4.2  Transient/sustained ratio shifts between block contexts
# ══════════════════════════════════════════════════════════════════════

# Per-cell TSI in contrast-context vs speed-context blocks
tsi_by_context = {'contrast': [], 'speed': [], 'subclass': [], 'cell_idx': []}

for ci_idx, ci in enumerate(matched_cells):
    for ctx_name, ctx_blocks in [('contrast', [0, 2]), ('speed', [1, 3])]:
        ctx_mask = np.isin(block_ids, ctx_blocks)
        ctx_trials = grating_trial_idx[ctx_mask]
        if len(ctx_trials) < 10:
            continue
        
        mean_psth = np.nanmean(dff_10hz[ctx_trials, :, ci], axis=0)  # (41,)
        early_r = np.nanmean(mean_psth[early_idx_t])
        late_r = np.nanmean(mean_psth[late_idx_t])
        tsi_val = (early_r - late_r) / (np.abs(early_r) + np.abs(late_r) + 1e-8)
        
        tsi_by_context[ctx_name].append(tsi_val)
        if ctx_name == 'contrast':
            tsi_by_context['subclass'].append(cell_subclasses[ci_idx])
            tsi_by_context['cell_idx'].append(ci)

tsi_ctx_df = pd.DataFrame({
    'TSI_contrast': tsi_by_context['contrast'],
    'TSI_speed': tsi_by_context['speed'],
    'subclass': tsi_by_context['subclass'],
    'subclass_short': [SUBCLASS_SHORT.get(s, s) for s in tsi_by_context['subclass']],
})

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# ── Scatter: TSI contrast vs TSI speed ──
ax = axes[0]
for sc in present_sc_b:
    m = tsi_ctx_df['subclass'] == sc
    ax.scatter(tsi_ctx_df.loc[m, 'TSI_contrast'], tsi_ctx_df.loc[m, 'TSI_speed'],
               alpha=0.4, s=15, color=SUBCLASS_COLORS[sc], label=SUBCLASS_SHORT[sc])
lims = [-1.5, 1.5]
ax.plot(lims, lims, 'k--', alpha=0.3)
ax.set_xlabel('TSI (Contrast Context)')
ax.set_ylabel('TSI (Speed Context)')
ax.set_title('C4: Transient/Sustained Index by Context', fontweight='bold')
ax.legend(fontsize=7)
ax.set_xlim(lims); ax.set_ylim(lims)

# ── Paired difference: TSI_contrast - TSI_speed ──
ax = axes[1]
tsi_ctx_df['TSI_diff'] = tsi_ctx_df['TSI_contrast'] - tsi_ctx_df['TSI_speed']
short_order_b = [SUBCLASS_SHORT[s] for s in present_sc_b]
short_pal_b = {SUBCLASS_SHORT[k]: v for k, v in SUBCLASS_COLORS.items()}
sns.violinplot(data=tsi_ctx_df, x='subclass_short', y='TSI_diff',
               order=short_order_b, palette=short_pal_b, cut=0, inner='box', ax=ax)
ax.axhline(0, color='k', ls='--', alpha=0.5)
ax.set_ylabel('ΔTSI (Contrast - Speed)')
ax.set_title('C4: Context-Dependent TSI Shift', fontweight='bold')
ax.set_xlabel('')
ax.tick_params(axis='x', rotation=45)

# ── Within-trial time course for contrast vs speed blocks (population mean) ──
ax = axes[2]
for ctx_name, ctx_blocks, color in [('Contrast-ctx', [0, 2], 'blue'), ('Speed-ctx', [1, 3], 'orange')]:
    ctx_mask = np.isin(block_ids, ctx_blocks)
    ctx_trials = grating_trial_idx[ctx_mask]
    mean_p = np.nanmean(dff_10hz[ctx_trials][:, :, matched_cells], axis=(0, 2))
    sem_p = np.nanstd(np.nanmean(dff_10hz[ctx_trials][:, :, matched_cells], axis=2), axis=0) / np.sqrt(len(ctx_trials))
    ax.fill_between(time_rel, mean_p - sem_p, mean_p + sem_p, alpha=0.2, color=color)
    ax.plot(time_rel, mean_p, color=color, linewidth=2, label=ctx_name)
ax.axvline(0, color='k', ls='--', alpha=0.4)
ax.axvline(2.0, color='k', ls=':', alpha=0.4)
ax.set_xlabel('Time (s)')
ax.set_ylabel('ΔF/F')
ax.set_title('C4: Population PSTH by Block Context', fontweight='bold')
ax.legend()

plt.suptitle('C4: Context Effects on Temporal Response Shape', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# ── Stats: paired Wilcoxon on TSI_contrast vs TSI_speed per subclass ──
from scipy.stats import wilcoxon
print("\n=== Paired Wilcoxon: TSI contrast vs speed context ===")
for sc in present_sc_b:
    m = tsi_ctx_df['subclass'] == sc
    c_vals = tsi_ctx_df.loc[m, 'TSI_contrast'].values
    s_vals = tsi_ctx_df.loc[m, 'TSI_speed'].values
    valid = ~np.isnan(c_vals) & ~np.isnan(s_vals)
    if valid.sum() >= 10:
        stat, p = wilcoxon(c_vals[valid], s_vals[valid])
        print(f"  {SUBCLASS_SHORT[sc]:10s}: W={stat:.0f}, p={p:.2e}, n={valid.sum()}")